# Employee Sentiment Analysis

> **Project:** Employee Sentiment & Engagement Analysis  
> **Dataset:** `test(in).csv` (Enron Email Corpus)  
> **Language:** Python  
> **Libraries:** PyTorch, HuggingFace Transformers, scikit-learn, pandas, matplotlib, seaborn  

---

## Table of Contents

1. [Setup & Imports](#setup)
2. [Task 1: Sentiment Labeling](#task1)
3. [Task 2: Exploratory Data Analysis](#task2)
4. [Task 3: Employee Score Calculation](#task3)
5. [Task 4: Employee Ranking](#task4)
6. [Task 5: Flight Risk Identification](#task5)
7. [Task 6: Predictive Modeling](#task6)
8. [Summary & Conclusions](#summary)


---

<a id='setup'></a>
## Setup & Imports

We begin by importing all required libraries and configuring the visualization style.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import re
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visual style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid', palette='husl')

# Visualization directory
VIZ_DIR = 'visualization'
os.makedirs(VIZ_DIR, exist_ok=True)

print('All libraries loaded successfully!')


### Data Loading & Cleaning

We load the raw `test(in).csv` dataset and perform several cleaning steps:
- Remove MIME/quoted-printable encoding artifacts from email bodies
- Parse date fields into datetime objects
- Extract employee names from email addresses
- Compute message length and word count features


In [ ]:
from data_prep import load_and_clean_data

df = load_and_clean_data('test(in).csv')
df.head()


In [ ]:
# Quick overview of the cleaned dataset
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Unique employees: {df["employee"].nunique()}')
df.info()


---

<a id='task1'></a>
## Task 1: Sentiment Labeling

### Approach & Justification

We use the **`cardiffnlp/twitter-roberta-base-sentiment-latest`** model from HuggingFace Transformers. This is a RoBERTa-based model fine-tuned on ~124M tweets for 3-class sentiment classification.

**Why this model?**
1. **Native 3-class output** — matches our Positive/Neutral/Negative requirement exactly
2. **Pre-trained and well-validated** — no labeled training data needed
3. **Handles informal/mixed-tone text well** — suitable for email communications
4. **Runs locally** — fully reproducible, no API keys or external dependencies
5. **Widely cited** — established model in NLP research

Each message's combined subject + body text (truncated to 500 characters) is processed through the model pipeline.


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load the sentiment model
model_name = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    truncation=True,
    device=-1  # CPU
)

# Label mapping
label_map = {
    'positive': 'Positive',
    'negative': 'Negative',
    'neutral': 'Neutral'
}

print('Model loaded successfully!')


In [ ]:
# Classify sentiment in batches
batch_size = 64
sentiments = []
total = len(df)

for i in range(0, total, batch_size):
    batch_texts = df['full_text'].iloc[i:i+batch_size].tolist()
    batch_texts = [str(t)[:500] if t and len(str(t).strip()) > 0 else 'neutral message' for t in batch_texts]
    
    try:
        results = sentiment_pipeline(batch_texts, batch_size=batch_size)
        for r in results:
            label = r['label'].lower()
            sentiments.append(label_map.get(label, 'Neutral'))
    except Exception as e:
        for t in batch_texts:
            try:
                r = sentiment_pipeline(t)[0]
                sentiments.append(label_map.get(r['label'].lower(), 'Neutral'))
            except:
                sentiments.append('Neutral')
    
    if (i // batch_size) % 25 == 0:
        print(f'Progress: {min(i + batch_size, total)}/{total} messages')

df['sentiment'] = sentiments

print(f'\nSentiment labeling complete!')
print(f'\nDistribution:')
print(df['sentiment'].value_counts())
print(f'\nNaN check: {df["sentiment"].isna().sum()} missing values')


In [ ]:
# Save labeled dataset
df.to_csv('labeled_data.csv', index=False)
print('Labeled dataset saved to labeled_data.csv')

# Show sample
df[['Subject', 'employee', 'date', 'sentiment']].head(10)


---

<a id='task2'></a>
## Task 2: Exploratory Data Analysis (EDA)

In this section, we explore the dataset's structure, distribution, and trends through comprehensive analysis and visualizations.


### 2.1 Dataset Overview


In [ ]:
# Dataset overview
print(f'Total records: {len(df)}')
print(f'\nData Types:')
print(df.dtypes)
print(f'\nMissing Values:')
print(df.isnull().sum())
print(f'\nBasic Statistics:')
df[['msg_length', 'word_count']].describe()


### 2.2 Sentiment Distribution

We visualize the distribution of sentiment labels across the dataset.


In [ ]:
# Sentiment distribution
colors = {'Positive': '#2ecc71', 'Neutral': '#3498db', 'Negative': '#e74c3c'}
sentiment_counts = df['sentiment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
bars = axes[0].bar(sentiment_counts.index, sentiment_counts.values,
                   color=[colors.get(s, '#95a5a6') for s in sentiment_counts.index],
                   edgecolor='white', linewidth=1.5)
axes[0].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for bar, count in zip(bars, sentiment_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                f'{count}\n({count/len(df)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%',
            colors=[colors.get(s, '#95a5a6') for s in sentiment_counts.index],
            startangle=90, explode=[0.05]*len(sentiment_counts),
            shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Sentiment Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.3 Sentiment Trends Over Time

We analyze how sentiment evolves month-by-month to identify temporal patterns.


In [ ]:
# Monthly sentiment trends
monthly_sentiment = df.groupby([df['date'].dt.to_period('M'), 'sentiment']).size().unstack(fill_value=0)
monthly_sentiment.index = monthly_sentiment.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 6))
for sent, color in colors.items():
    if sent in monthly_sentiment.columns:
        ax.plot(monthly_sentiment.index, monthly_sentiment[sent],
                label=sent, color=color, linewidth=2, marker='o', markersize=4)

ax.set_title('Monthly Sentiment Trends', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Message Count')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/sentiment_trends_over_time.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.4 Top Senders by Volume


In [ ]:
# Top senders
top_senders = df['employee'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(top_senders)), top_senders.values,
               color=sns.color_palette('viridis', len(top_senders)))
ax.set_yticks(range(len(top_senders)))
ax.set_yticklabels(top_senders.index)
ax.set_xlabel('Number of Messages')
ax.set_title('Top 15 Employees by Message Volume', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(top_senders.values):
    ax.text(v + 5, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/top_senders.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.5 Message Length Distribution


In [ ]:
# Message length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].hist(df['msg_length'].clip(upper=5000), bins=50, color='#3498db', alpha=0.7, edgecolor='white')
axes[0].set_title('Message Length Distribution (chars)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['msg_length'].median(), color='red', linestyle='--',
                label=f"Median: {df['msg_length'].median():.0f}")
axes[0].legend()

axes[1].hist(df['word_count'].clip(upper=1000), bins=50, color='#e67e22', alpha=0.7, edgecolor='white')
axes[1].set_title('Word Count Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(df['word_count'].median(), color='red', linestyle='--',
                label=f"Median: {df['word_count'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/message_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.6 Message Length by Sentiment


In [ ]:
# Box plot: word count by sentiment
fig, ax = plt.subplots(figsize=(10, 6))
sentiment_order = ['Positive', 'Neutral', 'Negative']
palette = [colors[s] for s in sentiment_order if s in df['sentiment'].unique()]
existing_order = [s for s in sentiment_order if s in df['sentiment'].unique()]

sns.boxplot(data=df[df['word_count'] < 500], x='sentiment', y='word_count',
            order=existing_order, palette=palette, ax=ax, showfliers=False)
ax.set_title('Word Count by Sentiment Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Word Count')
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/message_length_by_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.7 Day-of-Week Analysis


In [ ]:
# Sentiment by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_sentiment = df.groupby(['day_of_week', 'sentiment']).size().unstack(fill_value=0)
day_sentiment = day_sentiment.reindex(day_order)

fig, ax = plt.subplots(figsize=(12, 6))
day_sentiment.plot(kind='bar', stacked=True, ax=ax,
                   color=[colors.get(c, '#95a5a6') for c in day_sentiment.columns],
                   edgecolor='white', linewidth=0.5)
ax.set_title('Sentiment Distribution by Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Message Count')
ax.legend(title='Sentiment', frameon=True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/sentiment_by_day_of_week.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.8 Word Clouds by Sentiment


In [ ]:
# Word clouds per sentiment
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (sent, color_hex) in enumerate([('Positive', '#2ecc71'), ('Neutral', '#3498db'), ('Negative', '#e74c3c')]):
    text = ' '.join(df[df['sentiment'] == sent]['body_clean'].dropna().astype(str).tolist())
    if text.strip():
        wc = WordCloud(width=800, height=400, background_color='white',
                      max_words=100,
                      colormap='Greens' if sent == 'Positive' else ('Blues' if sent == 'Neutral' else 'Reds'),
                      contour_width=2, contour_color=color_hex).generate(text)
        axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(f'{sent} Messages', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/word_clouds.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.9 Anomaly Detection — Message Volume Spikes

We identify days with unusually high message volumes (> mean + 2σ) that may indicate significant organizational events.


In [ ]:
# Anomaly detection
daily_volume = df.groupby(df['date'].dt.date).size()
mean_vol = daily_volume.mean()
std_vol = daily_volume.std()
anomaly_threshold = mean_vol + 2 * std_vol
anomaly_dates = daily_volume[daily_volume > anomaly_threshold]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_volume.index, daily_volume.values, color='#3498db', alpha=0.6, linewidth=0.8)
ax.axhline(y=mean_vol, color='green', linestyle='--', label=f'Mean ({mean_vol:.0f})')
ax.axhline(y=anomaly_threshold, color='red', linestyle='--', label=f'Anomaly threshold ({anomaly_threshold:.0f})')
if len(anomaly_dates) > 0:
    ax.scatter(anomaly_dates.index, anomaly_dates.values, color='red', s=50, zorder=5, label='Anomalies')
ax.set_title('Daily Message Volume with Anomaly Detection', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Message Count')
ax.legend(frameon=True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Anomaly threshold (mean + 2σ): {anomaly_threshold:.0f} messages/day')
print(f'Anomalous days detected: {len(anomaly_dates)}')
if len(anomaly_dates) > 0:
    print(f'Top anomaly dates:')
    print(anomaly_dates.sort_values(ascending=False).head(5))


---

<a id='task3'></a>
## Task 3: Employee Score Calculation

### Scoring Method

We compute a monthly sentiment score per employee using the following mapping:

| Sentiment | Score |
|-----------|-------|
| Positive  | +1    |
| Neutral   |  0    |
| Negative  | −1    |

Scores are **aggregated per employee per calendar month** and **reset at the start of each new month** (no carry-over between months).


In [ ]:
# Map sentiment to numeric score
score_map = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
df['score'] = df['sentiment'].map(score_map)

# Aggregate by employee and month
monthly_scores = df.groupby(['employee', 'employee_email', 'year_month']).agg(
    total_score=('score', 'sum'),
    msg_count=('score', 'count'),
    positive_count=('score', lambda x: (x == 1).sum()),
    neutral_count=('score', lambda x: (x == 0).sum()),
    negative_count=('score', lambda x: (x == -1).sum())
).reset_index()

print(f'Monthly employee-month combinations: {len(monthly_scores)}')
monthly_scores.to_csv('monthly_scores.csv', index=False)
print('Saved: monthly_scores.csv')
monthly_scores.head(10)


---

<a id='task4'></a>
## Task 4: Employee Ranking

### Ranking Method

For each month, we identify:
- **Top 3 Positive Employees** — highest monthly sentiment scores
- **Top 3 Negative Employees** — lowest (most negative) monthly sentiment scores

Ties are broken alphabetically by employee name (ascending).


In [ ]:
# Generate monthly rankings
all_top_positive = []
all_top_negative = []
unique_months = sorted(monthly_scores['year_month'].unique())

for ym in unique_months:
    month_data = monthly_scores[monthly_scores['year_month'] == ym].copy()
    
    # Top 3 Positive: sort descending by score, then ascending by name
    month_sorted_pos = month_data.sort_values(by=['total_score', 'employee'], ascending=[False, True])
    top3_pos = month_sorted_pos.head(3)[['employee', 'year_month', 'total_score']].copy()
    top3_pos['rank_type'] = 'Top Positive'
    top3_pos['rank'] = range(1, len(top3_pos) + 1)
    
    # Top 3 Negative: sort ascending by score, then ascending by name
    month_sorted_neg = month_data.sort_values(by=['total_score', 'employee'], ascending=[True, True])
    top3_neg = month_sorted_neg.head(3)[['employee', 'year_month', 'total_score']].copy()
    top3_neg['rank_type'] = 'Top Negative'
    top3_neg['rank'] = range(1, len(top3_neg) + 1)
    
    all_top_positive.append(top3_pos)
    all_top_negative.append(top3_neg)

rankings_positive = pd.concat(all_top_positive, ignore_index=True)
rankings_negative = pd.concat(all_top_negative, ignore_index=True)

print('── TOP 3 POSITIVE EMPLOYEES (per month) ──')
for ym in unique_months:
    month_pos = rankings_positive[rankings_positive['year_month'] == ym]
    if len(month_pos) > 0:
        print(f'\n  {ym}:')
        for _, row in month_pos.iterrows():
            print(f'    #{int(row["rank"])} {row["employee"]} (score: {row["total_score"]})')


In [ ]:
print('── TOP 3 NEGATIVE EMPLOYEES (per month) ──')
for ym in unique_months:
    month_neg = rankings_negative[rankings_negative['year_month'] == ym]
    if len(month_neg) > 0:
        print(f'\n  {ym}:')
        for _, row in month_neg.iterrows():
            print(f'    #{int(row["rank"])} {row["employee"]} (score: {row["total_score"]})')


In [ ]:
# Visualization: Overall rankings
overall_scores = monthly_scores.groupby('employee')['total_score'].sum().sort_values(ascending=False)
top_10_pos_overall = overall_scores.head(10)
top_10_neg_overall = overall_scores.tail(10).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 10 Positive
axes[0].barh(range(len(top_10_pos_overall)), top_10_pos_overall.values,
             color=sns.color_palette('Greens_r', len(top_10_pos_overall)))
axes[0].set_yticks(range(len(top_10_pos_overall)))
axes[0].set_yticklabels(top_10_pos_overall.index)
axes[0].set_xlabel('Total Sentiment Score')
axes[0].set_title('Top 10 Most Positive Employees (Overall)', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
for i, v in enumerate(top_10_pos_overall.values):
    axes[0].text(v + 0.5, i, f'{v:.0f}', va='center', fontweight='bold')

# Top 10 Negative
axes[1].barh(range(len(top_10_neg_overall)), top_10_neg_overall.values,
             color=sns.color_palette('Reds', len(top_10_neg_overall)))
axes[1].set_yticks(range(len(top_10_neg_overall)))
axes[1].set_yticklabels(top_10_neg_overall.index)
axes[1].set_xlabel('Total Sentiment Score')
axes[1].set_title('Top 10 Most Negative Employees (Overall)', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()
for i, v in enumerate(top_10_neg_overall.values):
    axes[1].text(v - 0.5 if v < 0 else v + 0.5, i, f'{v:.0f}', va='center', fontweight='bold',
                ha='right' if v < 0 else 'left')

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/employee_rankings.png', dpi=150, bbox_inches='tight')
plt.show()

# Save rankings
rankings_positive.to_csv('rankings_positive.csv', index=False)
rankings_negative.to_csv('rankings_negative.csv', index=False)
print('Saved: rankings_positive.csv, rankings_negative.csv')


---

<a id='task5'></a>
## Task 5: Flight Risk Identification

### Definition

A **flight risk** is any employee who sent **4 or more negative messages within any rolling 30-day window**.

### Implementation

- For each employee, we collect all dates of negative messages (sorted chronologically)
- Using a **sliding window approach**, for each negative message date, we count how many other negative messages occurred within the next 30 days
- If any window contains ≥ 4 negative messages, the employee is flagged
- This handles overlapping windows correctly and is NOT limited to calendar month boundaries


In [ ]:
# Flight risk analysis
negative_msgs = df[df['sentiment'] == 'Negative'][['employee', 'employee_email', 'date']].copy()
negative_msgs = negative_msgs.sort_values(['employee', 'date'])

def check_flight_risk(dates, threshold=4, window_days=30):
    """Check if any 30-day rolling window has >= threshold negative messages."""
    dates_sorted = sorted(dates)
    n = len(dates_sorted)
    if n < threshold:
        return False, n, None
    
    max_count = 0
    worst_window_start = None
    
    for i in range(n):
        window_end = dates_sorted[i] + pd.Timedelta(days=window_days)
        count = sum(1 for d in dates_sorted[i:] if d <= window_end)
        if count > max_count:
            max_count = count
            worst_window_start = dates_sorted[i]
    
    return max_count >= threshold, max_count, worst_window_start

# Analyze each employee
flight_risk_results = []
for employee, group in negative_msgs.groupby('employee'):
    email = group['employee_email'].iloc[0]
    dates = group['date'].tolist()
    is_risk, max_neg, window_start = check_flight_risk(dates)
    flight_risk_results.append({
        'employee': employee,
        'employee_email': email,
        'total_negative_msgs': len(dates),
        'max_negatives_in_30d_window': max_neg,
        'is_flight_risk': is_risk,
        'worst_window_start': window_start
    })

flight_risk_df = pd.DataFrame(flight_risk_results)
flagged = flight_risk_df[flight_risk_df['is_flight_risk'] == True].sort_values(
    'max_negatives_in_30d_window', ascending=False)

print(f'Total employees analyzed: {len(flight_risk_df)}')
print(f'Employees flagged as FLIGHT RISK: {len(flagged)}')
print()
if len(flagged) > 0:
    print(flagged[['employee', 'employee_email', 'total_negative_msgs', 'max_negatives_in_30d_window']].to_string(index=False))
else:
    print('No employees meet the flight risk threshold.')

flight_risk_df.to_csv('flight_risk_results.csv', index=False)


In [ ]:
# Flight risk visualization
fig, ax = plt.subplots(figsize=(12, max(6, len(flagged) * 0.5 + 2)))

if len(flagged) > 0:
    top_risks = flagged.head(20)
    ax.barh(range(len(top_risks)), top_risks['max_negatives_in_30d_window'].values,
            color='#e74c3c', edgecolor='white')
    ax.set_yticks(range(len(top_risks)))
    ax.set_yticklabels(top_risks['employee'].values)
    ax.set_xlabel('Max Negative Messages in Any 30-Day Window')
    ax.set_title('Flight Risk Employees\n(≥4 negative messages in any 30-day window)',
                fontsize=14, fontweight='bold')
    ax.axvline(x=4, color='red', linestyle='--', linewidth=2, label='Threshold (4)')
    ax.legend()
    ax.invert_yaxis()
    for i, v in enumerate(top_risks['max_negatives_in_30d_window'].values):
        ax.text(v + 0.1, i, str(v), va='center', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No employees flagged as flight risk',
            ha='center', va='center', fontsize=16, transform=ax.transAxes)
    ax.set_title('Flight Risk Employees', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/flight_risk.png', dpi=150, bbox_inches='tight')
plt.show()


---

<a id='task6'></a>
## Task 6: Predictive Modeling (Linear Regression)

### Objective

Build a linear regression model to predict monthly employee sentiment scores.

### Features
- `msg_count` — Total messages sent that month
- `avg_msg_length` — Average message length (characters)
- `avg_word_count` — Average word count per message
- `total_words` — Total words written that month
- `positive_ratio` — Fraction of positive messages
- `negative_ratio` — Fraction of negative messages

### Target
- `total_score` — Monthly sentiment score


In [ ]:
# Build feature matrix
features_df = df.groupby(['employee', 'year_month']).agg(
    total_score=('score', 'sum'),
    msg_count=('score', 'count'),
    avg_msg_length=('msg_length', 'mean'),
    avg_word_count=('word_count', 'mean'),
    total_words=('word_count', 'sum'),
    positive_count=('score', lambda x: (x == 1).sum()),
    negative_count=('score', lambda x: (x == -1).sum()),
).reset_index()

features_df['positive_ratio'] = features_df['positive_count'] / features_df['msg_count']
features_df['negative_ratio'] = features_df['negative_count'] / features_df['msg_count']

feature_cols = ['msg_count', 'avg_msg_length', 'avg_word_count', 'total_words',
                'positive_ratio', 'negative_ratio']
target_col = 'total_score'

X = features_df[feature_cols].fillna(0)
y = features_df[target_col]

print(f'Feature matrix shape: {X.shape}')
print(f'Target stats: mean={y.mean():.2f}, std={y.std():.2f}')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')


In [ ]:
# Train linear regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
y_pred_train = lr_model.predict(X_train)
y_pred_test = lr_model.predict(X_test)

# Metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print('── MODEL PERFORMANCE ──')
print(f'  Training R²:  {r2_train:.4f}')
print(f'  Test R²:      {r2_test:.4f}')
print(f'  Test MAE:     {mae_test:.4f}')
print(f'  Test RMSE:    {rmse_test:.4f}')

print(f'\n── FEATURE COEFFICIENTS ──')
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))
print(f'\nIntercept: {lr_model.intercept_:.4f}')


### Model Interpretation

The coefficients reveal the relationship between each feature and the predicted sentiment score:
- **Positive ratio** and **negative ratio** have the strongest coefficients, as they directly reflect sentiment composition
- **Message count** and **word count** features capture communication intensity
- The model provides a baseline for understanding which employee behaviors predict sentiment trends


In [ ]:
# Model performance visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_test, alpha=0.5, color='#3498db', s=30)
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Score', fontsize=12)
axes[0].set_ylabel('Predicted Score', fontsize=12)
axes[0].set_title(f'Actual vs Predicted\n(R² = {r2_test:.4f})', fontsize=13, fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.5, color='#e67e22', s=30)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Score', fontsize=12)
axes[1].set_ylabel('Residual', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=13, fontweight='bold')

# Feature importance
coef_sorted = coef_df.sort_values('Coefficient')
bar_colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_sorted['Coefficient']]
axes[2].barh(coef_sorted['Feature'], coef_sorted['Coefficient'], color=bar_colors, edgecolor='white')
axes[2].set_xlabel('Coefficient Value', fontsize=12)
axes[2].set_title('Feature Importance', fontsize=13, fontweight='bold')
axes[2].axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/model_performance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Residual distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(residuals, bins=30, color='#9b59b6', alpha=0.7, edgecolor='white')
ax.set_xlabel('Residual Value')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Prediction Residuals', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.axvline(x=residuals.mean(), color='blue', linestyle='--', label=f'Mean: {residuals.mean():.3f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/residual_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


---

<a id='summary'></a>
## Summary & Conclusions

### Key Findings

1. **Sentiment Distribution:** The analysis reveals the overall tone of employee communications across the organization.

2. **Top Performers:** Employees with consistently positive scores demonstrate constructive communication patterns that can serve as organizational models.

3. **At-Risk Employees:** The flight risk analysis identifies employees with concentrated negative messaging patterns, enabling proactive intervention.

4. **Predictive Insights:** The linear regression model reveals which message characteristics most strongly correlate with sentiment, enabling data-driven monitoring.

### Recommendations

- **Proactive Engagement:** Prioritize check-ins with flight-risk employees
- **Best Practices:** Study communication patterns of top-positive employees
- **Continuous Monitoring:** Update the model with new data for improved accuracy
- **Organizational Events:** Correlate sentiment trends with known events for context


In [ ]:
# Final summary
print('=' * 60)
print('ANALYSIS COMPLETE')
print('=' * 60)
print(f'\nTotal messages analyzed: {len(df):,}')
print(f'Unique employees: {df["employee"].nunique()}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'\nSentiment Distribution:')
for sent, count in df['sentiment'].value_counts().items():
    print(f'  {sent}: {count:,} ({count/len(df)*100:.1f}%)')
print(f'\nModel R²: {r2_test:.4f}')
print(f'Flight risk employees: {len(flagged)}')
